In [1]:
%load_ext pycograd

In [2]:
import numpy as np
from pycograd import capture, cross_entropy, grad, optimize, relu, softmax, value_and_grad

rng = np.random.default_rng(42)

m = 40
centers = np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.5, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]   # one-hot, 3 classes

In [3]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    logits = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    forward = optimize(capture($ |> logits |> softmax, X))
    loss = lambda x, y: x |> logits |> cross_entropy($, y)
    optimized = optimize(value_and_grad(capture(loss, X, Y)))
    obj = |> X |> logits |> cross_entropy($, Y)
    for _i in range(10):
        _values, grads = optimized(X, Y)
        weights.step(grads, 0.5)

In [4]:
np.all(np.argmax(forward(X), axis=1) == labels)

np.True_

In [5]:
# Eager alternative to the compiled `value_and_grad(capture(...))` step above:
# `weights.grad(obj)` differentiates an ambient `|>` objective directly and hands back a
# ParamDict -- the very thing `weights.step` consumes, no capture/optimize needed. Run it
# inside `with weights:` so the w1/b1/... proxies are live.
with weights:
    obj = |> X |> logits |> cross_entropy($, Y)
    value, grads = weights.grad(obj)

float(value), {name: g.shape for name, g in grads.items()}

(0.03296407279607119, {'w1': (2, 16), 'b1': (16,), 'w2': (16, 3), 'b2': (3,)})

In [6]:
# `jit=True` compiles the gradient into pycograd's own optimized forward+backward graph
# (capture -> grad_graph -> optimize) the first time, caches it on `weights`, and replays it
# each call -- the manual `optimize(value_and_grad(capture(...)))` pattern from the training
# cell, now behind a flag. On the numpy backend the objective takes its data positionally
# (so the trace flows through the weights); the result matches the eager path.
with weights:
    loss = lambda x, y: x |> logits |> cross_entropy($, y)
    v_eager, g_eager = weights.grad(loss, X, Y)            # eager numpy tape
    v_jit, g_jit = weights.grad(loss, X, Y, jit=True)      # compiled once, cached on `weights`
    _ = weights.grad(loss, X, Y, jit=True)                 # reuses the cached graph

# same value and same per-weight gradients as the eager path:
bool(np.isclose(v_eager, v_jit)), {k: bool(np.allclose(g_eager[k], g_jit[k])) for k in weights}

(True, {'w1': True, 'b1': True, 'w2': True, 'b2': True})